In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

workflow_end_time = datetime.now()

run_id = dbutils.jobs.taskContext().taskRunId()

audit_df = spark.table(
    "helathcare_audit.audit_table.Patient_load_details"
).filter(
    (F.col("run_id") == str(run_id))
    &
    (F.col("record_type") == "NOTEBOOK")
)

total_tasks = audit_df.count()

failed_tasks = audit_df.filter(
    F.col("status") == "FAILED"
).count()

successful_tasks = audit_df.filter(
    F.col("status") == "SUCCESS"
).count()

workflow_status = (
    "FAILED"
    if failed_tasks > 0
    else "SUCCESS"
)

workflow_start_time = audit_df.agg(
    F.min("start_time")
).first()[0]

total_records = audit_df.agg(
    F.sum("record_count")
).first()[0]

write_audit(
    target_table="WORKFLOW",
    run_id=run_id,
    record_count=total_records,
    start_time=workflow_start_time,
    end_time=workflow_end_time,
    status=workflow_status,
    error_message=None,
    record_type="WORKFLOW"
)